In [3]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
from PIL import Image

# 작업 경로 동적 할당
BASE_DIR = os.path.abspath("") # 주피터 환경에 맞게 현재 경로 설정
EXTRACT_DIR = os.path.join(BASE_DIR, "dataset")

# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (128, 128)
EPOCHS = 100
LEARNING_RATE = 0.001

print("라이브러리 로드 및 설정 완료")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
zip_files = ["imagenet_cats.zip", "imagenet_dogs.zip", "imagenet_horse.zip", "imagenet_zebra.zip"]

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)

for zip_file in zip_files:
    zip_path = os.path.join(BASE_DIR, zip_file)
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            class_name = zip_file.replace(".zip", "")
            class_dir = os.path.join(EXTRACT_DIR, class_name)
            if not os.path.exists(class_dir):
                os.makedirs(class_dir)
            zip_ref.extractall(class_dir)

# 파일 확장자가 아닌 실제 이미지 헤더 검증 (WHY: 가짜 이미지로 인한 학습 중단 방지)
print("데이터 딥 클렌징을 시작합니다...")
removed_count = 0

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            if os.path.getsize(file_path) == 0:
                raise IOError("Zero byte file")
            with Image.open(file_path) as img:
                img.verify() 
        except Exception:
            os.remove(file_path)
            removed_count += 1

print(f"전처리 완료: 내부가 손상된 총 {removed_count}개의 불량 파일이 제거되었습니다.")

데이터 딥 클렌징을 시작합니다...
전처리 완료: 내부가 손상된 총 2002개의 불량 파일이 제거되었습니다.


In [ ]:
# 검증을 마친 순수 이미지 데이터셋 로딩
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    EXTRACT_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    EXTRACT_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"분류할 클래스: {class_names}")

# 데이터 로딩 병목 현상 방지를 위한 프리페치(Prefetch) 적용
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 4455 files belonging to 4 classes.
Using 3564 files for training.
Found 4455 files belonging to 4 classes.
Using 891 files for validation.
분류할 클래스: ['imagenet_cats', 'imagenet_dogs', 'imagenet_horse', 'imagenet_zebra']


In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

def train_and_evaluate(model, model_name):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy', metrics=['accuracy']
    )
    
    print(f"\n--- {model_name} 학습 시작 ---")
    history = model.fit(
        train_dataset, validation_data=val_dataset,
        epochs=EPOCHS, callbacks=[early_stopping], verbose=1
    )
    
    y_true, y_pred_probs = [], []
    for images, labels in val_dataset:
        y_true.extend(labels.numpy())
        preds = model.predict(images, verbose=0)
        y_pred_probs.extend(preds)
        
    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    
    print(f"\n[{model_name} 성능 평가 지표]")
    print(classification_report(y_true, y_pred_classes, target_names=class_names))
    
    y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    auroc = roc_auc_score(y_true_bin, y_pred_probs, multi_class='ovr') if NUM_CLASSES > 2 else roc_auc_score(y_true, y_pred_probs[:, 1])
    print(f"{model_name} AUROC: {auroc:.4f}\n")
    return history

In [ ]:
mlp_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1./255), 
    layers.Flatten(), # 2차원 공간 정보 파괴
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5), 
    layers.Dense(256, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

mlp_history = train_and_evaluate(mlp_model, "MLP")  


--- MLP 학습 시작 ---
Epoch 1/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 12s 78ms/step - accuracy: 0.2840 - loss: 6.8637 - val_accuracy: 0.3255 - val_loss: 1.3709
Epoch 2/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.3013 - loss: 1.3778 - val_accuracy: 0.3389 - val_loss: 1.3668
Epoch 3/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.3137 - loss: 1.3607 - val_accuracy: 0.3614 - val_loss: 1.3416
Epoch 4/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.3103 - loss: 1.3643 - val_accuracy: 0.3322 - val_loss: 1.3604
Epoch 5/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.3084 - loss: 1.3618 - val_accuracy: 0.3648 - val_loss: 1.3348
Epoch 6/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.3143 - loss: 1.3516 - val_accuracy: 0.3547 - val_loss: 1.3450
Epoch 7/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.2943 - loss: 1.3831 - val_accuracy: 0.3210 - val_loss: 1.3748
Epoch 8/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.29

c:\Users\fudal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\fudal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\fudal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), 
    layers.Dense(NUM_CLASSES, activation='softmax')
])

cnn_history = train_and_evaluate(cnn_model, "CNN")


--- CNN 학습 시작 ---
Epoch 1/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 18s 141ms/step - accuracy: 0.4568 - loss: 1.1608 - val_accuracy: 0.5859 - val_loss: 0.9584
Epoch 2/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 136ms/step - accuracy: 0.6038 - loss: 0.9249 - val_accuracy: 0.6049 - val_loss: 0.9084
Epoch 3/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 136ms/step - accuracy: 0.6650 - loss: 0.7981 - val_accuracy: 0.7295 - val_loss: 0.7030
Epoch 4/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 132ms/step - accuracy: 0.7116 - loss: 0.6989 - val_accuracy: 0.7520 - val_loss: 0.6761
Epoch 5/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 133ms/step - accuracy: 0.7497 - loss: 0.6159 - val_accuracy: 0.7531 - val_loss: 0.6501
Epoch 6/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 133ms/step - accuracy: 0.7893 - loss: 0.5229 - val_accuracy: 0.7710 - val_loss: 0.6378
Epoch 7/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 15s 132ms/step - accuracy: 0.8185 - loss: 0.4602 - val_accuracy: 0.7553 - val_loss: 0.7982
Epoch 8/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 20s 181ms/step -